# Assignment 04 — Fresh notebooks and labeled pandas data

Complete this notebook in local Jupyter or VS Code. Colab is not yet a supported assignment-submission path. Before submission, save, restart the kernel, and run all cells from top to bottom. Stored output is ignored by grading; the grader executes a fresh disposable copy. Generated CSV files are separate artifacts.

Do not edit the supplied setup cell. It verifies the immutable synthetic fixture without an absolute path, upload, network request, or Drive mount. Never place credentials or private information in notebook source or output.

In [ ]:
from hashlib import sha256
import json
from pathlib import Path

import numpy as np
import pandas as pd


def _find_assignment_base(start):
    current = start.resolve()
    while True:
        for candidate in (current, current / "04" / "assignment"):
            data_path = candidate / "data" / "purchases.csv"
            manifest_path = candidate / "data" / "fixture.json"
            if data_path.is_file() and manifest_path.is_file():
                return candidate
        if current.parent == current:
            return None
        current = current.parent


ASSIGNMENT_BASE = _find_assignment_base(Path.cwd())
assert ASSIGNMENT_BASE is not None, (
    "Could not find data/purchases.csv and data/fixture.json from this launch directory."
)

DATA_PATH = ASSIGNMENT_BASE / "data" / "purchases.csv"
MANIFEST_PATH = ASSIGNMENT_BASE / "data" / "fixture.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

required_manifest_fields = {
    "fixture_id",
    "provenance",
    "row_count",
    "columns",
    "sha256",
}
assert set(manifest) == required_manifest_fields, "Unexpected fixture manifest fields."
assert isinstance(manifest["fixture_id"], str) and manifest["fixture_id"], (
    "The fixture manifest needs a non-empty fixture_id."
)
assert manifest["provenance"] == "course-authored synthetic teaching data"
actual_sha256 = sha256(DATA_PATH.read_bytes()).hexdigest()
assert actual_sha256 == manifest["sha256"], "purchases.csv checksum mismatch."

fixture_preview = pd.read_csv(DATA_PATH)
assert list(fixture_preview.columns) == manifest["columns"]
assert len(fixture_preview) == manifest["row_count"]

OUTPUT_DIR = ASSIGNMENT_BASE / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LABELED_OUTPUT_PATH = OUTPUT_DIR / "labeled_block.csv"
SELECTED_OUTPUT_PATH = OUTPUT_DIR / "selected_purchases.csv"

print("Fixture:", manifest["fixture_id"])
print("Input:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## Task 1 — Repair visible dependency order

The next two code cells are deliberately out of order. Move the complete producer cell above the dependent cell. Do not duplicate the definition. After a restart-and-run-all, `base_rate` must be `3` and `adjusted_rate` must be `5`.

In [ ]:
adjusted_rate = base_rate + 2
print("adjusted_rate:", adjusted_rate)

In [ ]:
base_rate = 3
print("base_rate:", base_rate)

### State explanation

TODO: In a short paragraph, distinguish visible cell order, execution order, retained kernel state, and stored output. Explain the repair and why restart-and-run-all is the evidence that it worked.

## Task 2 — Construct and select labeled pandas objects

Keep the supplied arrays. Replace each `None` and TODO with the exact Series, DataFrame, bracket, `.loc`, and `.iloc` operations described in `README.md`. Preserve the named row index when writing `labeled_block.csv`.

In [ ]:
reading_values = np.array([12.5, 15.0, 11.5, 15.5])

reading_by_site = None  # TODO: create the labeled Series

measurement_values = np.array(
    [
        [12, 18],
        [15, 23],
        [10, 17],
        [15, 23],
    ]
)

measurement_table = None  # TODO: create the labeled DataFrame
# TODO: name its index record_id

baseline_series = None  # TODO: one bracket label -> Series
baseline_table = None  # TODO: a list inside brackets -> DataFrame

label_block = None  # TODO: .loc from site-102 through site-103
position_block = None  # TODO: equivalent .iloc selection

print("Series metadata:", reading_by_site.index, reading_by_site.dtype, reading_by_site.name)
print("DataFrame metadata:", measurement_table.index, measurement_table.columns)
print("shape:", measurement_table.shape)
print("dtypes:")
print(measurement_table.dtypes)
print("bracket return types:", type(baseline_series), type(baseline_table))

pd.testing.assert_frame_equal(label_block, position_block)
label_block.to_csv(LABELED_OUTPUT_PATH)
print("wrote:", LABELED_OUTPUT_PATH)

## Task 3 — Portable CSV round trip

Read only through `DATA_PATH`. Build the exact named mask, select explicit source columns with `.loc`, copy, derive `line_total`, sort with the unique tie-breaker, write with `index=False`, and read the file back into `round_trip`.

In [ ]:
purchases = None  # TODO: pd.read_csv(DATA_PATH)

# TODO: inspect shape, columns, dtypes, and head

quantity_at_least_two = None  # TODO: purchases["quantity"] >= 2

selected_purchases = None  # TODO: .loc mask and explicit source columns, then copy
# TODO: add line_total = quantity * unit_price
# TODO: sort by line_total descending, then purchase_id ascending
# TODO: write SELECTED_OUTPUT_PATH with index=False

round_trip = None  # TODO: read SELECTED_OUTPUT_PATH back with pandas
round_trip

## Final fresh-run verification

This supplied check gives immediate feedback. The grader does not trust this editable cell or its stored output; it appends independent verification to a disposable copy.

In [ ]:
assert base_rate == 3
assert adjusted_rate == 5
assert isinstance(reading_by_site, pd.Series)
assert isinstance(measurement_table, pd.DataFrame)
assert isinstance(baseline_series, pd.Series)
assert isinstance(baseline_table, pd.DataFrame)
pd.testing.assert_frame_equal(label_block, position_block)

expected_columns = [
    "purchase_id",
    "item",
    "quantity",
    "unit_price",
    "line_total",
]
assert list(round_trip.columns) == expected_columns
assert len(round_trip) == int(quantity_at_least_two.sum())
assert (round_trip["quantity"] >= 2).all()
assert np.allclose(
    round_trip["line_total"],
    round_trip["quantity"] * round_trip["unit_price"],
)
expected_order = round_trip.sort_values(
    by=["line_total", "purchase_id"],
    ascending=[False, True],
)["purchase_id"].tolist()
assert round_trip["purchase_id"].tolist() == expected_order

print("Assignment 04 fresh-run verification passed")